<a href="https://colab.research.google.com/github/RedHair11/Office-Hours/blob/main/Neural_Model_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# -----------------------------
# Import Required Libraries HELLO
# -----------------------------
import os                      # For file existence checks and path handling
import numpy as np             # For numerical array manipulation
import tensorflow as tf        # Core deep learning framework
import matplotlib.pyplot as plt # For plotting training performance

# Import Keras components for model building
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization, LayerNormalization,
    Conv1D, Add, GlobalAveragePooling1D, Activation
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# =============================================================================
# 1. DATA LOADING AND PREPROCESSING
# =============================================================================

def load_signal_data(data_dir=".", signals_name="signals.npy", labels_name="labels.npy",
                     samples=2000, time_steps=512, channels=8, seed=42):

    # Loads EEG/ECG-like signal data. If the files are not found,
    # it generates a synthetic dataset so the model runs end-to-end.

    rng = np.random.default_rng(seed)  # Random number generator for reproducibility

    sig_path = os.path.join(data_dir, signals_name)  # Path to signal file
    lab_path = os.path.join(data_dir, labels_name)   # Path to label file

    # Check if dataset files exist
    if os.path.isfile(sig_path) and os.path.isfile(lab_path):
        # Load pre-saved dataset
        signals = np.load(sig_path)  # Shape (N, T, C)
        labels = np.load(lab_path)   # Shape (N,)
    else:
        # --------------------------------------------------------------
        # SYNTHETIC DATA GENERATION (for demonstration and testing)
        # --------------------------------------------------------------
        N, T, C = samples, time_steps, channels
        t = np.linspace(0, 1, T, endpoint=False)

        # Generate baseline "healthy" signals (class 0)
        base0 = []
        for _ in range(N // 2):
            comp = 0
            # Create sum of multiple sine waves across frequencies
            for c in range(C):
                f = rng.uniform(3, 20)  # Frequency in Hz
                comp += np.sin(2*np.pi*f*t + rng.uniform(0, 2*np.pi)) * rng.uniform(0.3, 1.0)
            # Add mild Gaussian noise
            comp += rng.normal(0, 0.2, size=T)
            base0.append(comp)
        base0 = np.stack(base0)

        # Generate "pathological" signals (class 1)
        base1 = []
        for _ in range(N - N // 2):
            comp = 0
            for c in range(C):
                f = rng.uniform(3, 20)
                comp += np.sin(2*np.pi*f*t + rng.uniform(0, 2*np.pi)) * rng.uniform(0.3, 1.0)
            comp += rng.normal(0, 0.25, size=T)
            # Add random burst spikes to simulate abnormal events
            for _ in range(rng.integers(1, 4)):
                center = rng.integers(50, T-50)
                width = rng.integers(5, 30)
                amp = rng.uniform(1.0, 2.0)
                comp[center:center+width] += amp * np.hanning(width)
            base1.append(comp)
        base1 = np.stack(base1)

        # Function to add independent noise per channel
        def multichannel(stack):
            out = []
            for _ in range(C):
                out.append(stack + rng.normal(0, 0.05, size=stack.shape))
            return np.stack(out, axis=-1)  # Shape: (N, T, C)

        # Create full multichannel dataset
        X0 = multichannel(base0)
        X1 = multichannel(base1)
        signals = np.concatenate([X0, X1], axis=0)
        labels = np.concatenate([np.zeros(X0.shape[0]), np.ones(X1.shape[0])], axis=0)

        # Shuffle dataset for randomization
        perm = rng.permutation(signals.shape[0])
        signals, labels = signals[perm], labels[perm]

    # --------------------------------------------------------------
    # Normalize each signal window individually (zero mean, unit var)
    # --------------------------------------------------------------
    mu = signals.mean(axis=(1, 2), keepdims=True)
    sd = signals.std(axis=(1, 2), keepdims=True) + 1e-7
    signals = (signals - mu) / sd

    return signals.astype(np.float32), labels.astype(np.float32)

# Load dataset (either real or synthetic)
signals, labels = load_signal_data()

# =============================================================================
# 2. DEFINE QUANTUM-INSPIRED LAYERS
# =============================================================================

class PhaseEncoding(tf.keras.layers.Layer):

    # Quantum-inspired phase encoding layer.
    # Converts real inputs into a higher-dimensional phase representation
    # by mapping x → [cos(sx), sin(sx)], similar to quantum state encoding.

    def __init__(self, scale=1.0, **kwargs):
        super().__init__(**kwargs)
        # Learnable scaling factor (trainable phase amplitude)
        self.scale = tf.Variable(initial_value=scale, trainable=True, dtype=tf.float32, name="phase_scale")

    def call(self, inputs):
        # Scale input by the learnable factor
        x = self.scale * inputs
        # Compute cosine and sine components
        c = tf.math.cos(x)
        s = tf.math.sin(x)
        # Concatenate both to form a doubled feature vector (cos, sin)
        z = tf.concat([c, s], axis=-1)
        return z  # Shape: (batch, time_steps, 2*channels)

class PseudoUnitaryBlock(tf.keras.layers.Layer):

    # Approximates a unitary transformation using real-valued 2D rotations.
    # - Mimics quantum circuit layers (unitary gates) via learnable angles.
    # - Each rotation preserves vector magnitude (energy).

    def __init__(self, num_stacks=2, **kwargs):
        super().__init__(**kwargs)
        self.num_stacks = num_stacks  # Number of rotation layers

    def build(self, input_shape):
        F = int(input_shape[-1])  # Feature dimension
        self.F = F
        # If feature count is odd, pad with one dummy dimension
        self.needs_pad = (F % 2 == 1)
        self.F_eff = F + 1 if self.needs_pad else F
        self.num_pairs = self.F_eff // 2

        # Create learnable rotation angles θ for each pair in each stack
        self.theta_vars = []
        for k in range(self.num_stacks):
            theta = self.add_weight(
                name=f"theta_stack{k}",
                shape=(self.num_pairs,),
                initializer=tf.keras.initializers.RandomUniform(-0.1, 0.1),
                trainable=True
            )
            self.theta_vars.append(theta)

    def call(self, inputs):
        x = inputs
        # Pad input if needed
        if self.needs_pad:
            pad = tf.zeros_like(x[..., :1])
            x = tf.concat([x, pad], axis=-1)

        # Apply stacked rotation layers
        for theta in self.theta_vars:
            c = tf.math.cos(theta)
            s = tf.math.sin(theta)
            # Reshape to split features into pairs (num_pairs, 2)
            x_pairs = tf.reshape(x, tf.concat([tf.shape(x)[:-1], [self.num_pairs, 2]], axis=0))
            x0 = x_pairs[..., 0]
            x1 = x_pairs[..., 1]
            # Apply rotation matrix [ [cos, -sin], [sin, cos] ]
            x0p = c * x0 - s * x1
            x1p = s * x0 + c * x1
            # Stack rotated pairs back
            x_pairs_rot = tf.stack([x0p, x1p], axis=-1)
            x = tf.reshape(x_pairs_rot, tf.concat([tf.shape(x)[:-2], [self.F_eff]], axis=0))

        # Remove padded channel if added
        if self.needs_pad:
            x = x[..., :self.F]
        return x

# =============================================================================
# 3. BUILD THE HYBRID MODEL
# =============================================================================

# Define input shape (e.g., 512 time steps × 8 channels)
input_shape = signals.shape[1:]
inputs = Input(shape=input_shape, name="input_window")

# ----- Quantum-inspired front-end -----
# 1. Phase encoding: expand each channel into (cos, sin) representation
x = PhaseEncoding(scale=1.0, name="phase_encoding")(inputs)
# 2. Pseudo-unitary transform: energy-preserving mixing between features
x = PseudoUnitaryBlock(num_stacks=3, name="pseudo_unitary")(x)
# 3. Normalize and regularize
x = LayerNormalization(name="qi_layernorm")(x)
x = Dropout(0.1, name="qi_dropout")(x)

# ----- Temporal Convolutional Encoder -----
# Captures local dependencies and time patterns (similar to CNN on time series)
res = x  # Store skip connection
x = Conv1D(64, kernel_size=7, padding="same", activation="relu", name="tconv_1")(x)
x = LayerNormalization(name="tln_1")(x)
x = Conv1D(64, kernel_size=7, padding="same", activation="relu", name="tconv_2")(x)
x = LayerNormalization(name="tln_2")(x)
# Add skip (residual connection)
if res.shape[-1] != x.shape[-1]:
    res = Conv1D(64, kernel_size=1, padding="same", name="res_proj")(res)
x = Add(name="tconv_residual_sum")([x, res])
x = Activation("relu", name="tconv_residual_relu")(x)

# Global pooling: compress temporal information into a single vector
x = GlobalAveragePooling1D(name="gap")(x)

# ----- Dense (classical) classification head -----
# Fully connected layers for decision-level fusion
x = Dense(128, activation="relu", name="head_dense1")(x)
x = Dropout(0.3, name="head_dropout")(x)
x = Dense(64, activation="relu", name="head_dense2")(x)
output = Dense(1, activation="sigmoid", name="out_bin")(x)  # Binary classification output

# Combine all into one model
model = Model(inputs, output, name="QuantumInspiredHybridNet")
model.summary()



# =============================================================================
# 4. COMPILE AND TRAIN MODEL
# =============================================================================

# Compile model with Adam optimizer and standard binary cross-entropy loss
# Include AUROC (area under ROC curve) for better insight on performance
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auroc")]
)

# Split dataset into training and testing subsets
X_train, X_test, y_train, y_test = train_test_split(
    signals, labels, test_size=0.2, random_state=42, stratify=(labels > 0.5).astype(int)
)

# Define callbacks for early stopping and model checkpointing
os.makedirs("./checkpoints_qi", exist_ok=True)
ckpt_path = "./checkpoints_qi/qi_hybrid_best.keras"
callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    ModelCheckpoint(ckpt_path, monitor="val_auroc", mode="max", save_best_only=True, verbose=1)
]
# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=64,
    verbose=2,
    callbacks=callbacks
)
# =============================================================================
# 5. EVALUATION AND VISUALIZATION
# =============================================================================

# Evaluate model performance on the test set
eval_results = model.evaluate(X_test, y_test, verbose=0)
print("\nTest Results:")
for name, val in zip(model.metrics_names, eval_results):
    print(f"  {name}: {val:.4f}")

# Compute AUROC again using sklearn for confirmation
y_pred_prob = model.predict(X_test, verbose=0).ravel()
auroc = roc_auc_score(y_test, y_pred_prob)
print(f"[sklearn] AUROC: {auroc:.4f}")

# Plot training loss and AUROC across epochs
plt.figure(figsize=(12, 4))

# Plot training vs validation loss
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Binary Cross-Entropy Loss")
plt.xlabel("Epochs"); plt.ylabel("Loss")
plt.legend()

# Plot AUROC evolution
plt.subplot(1, 2, 2)
plt.plot(history.history["auroc"], label="Train AUROC")
plt.plot(history.history["val_auroc"], label="Val AUROC")
plt.title("Area Under ROC Curve")
plt.xlabel("Epochs"); plt.ylabel("AUROC")
plt.legend()

plt.tight_layout()
plt.show()

# =============================================================================
# Notes:
# - PhaseEncoding → simulates quantum state encoding using cos/sin embedding.
# - PseudoUnitaryBlock → acts like a quantum gate with energy-preserving 2D rotations.
# - CNN layers → handle temporal patterns (spatial/temporal correlations).
# - Dense head → performs classical decision-level classification.
# - For regression: replace final sigmoid with linear activation.
# =============================================================================


ValueError: Input 0 of layer "tconv_1" is incompatible with the layer: expected min_ndim=3, found ndim=1. Full shape received: (16,)